# Finish Position Model - Exploration

Baseline: MAE 2.97 (val), 3.32 (test) | Spearman 0.750 (val), 0.661 (test)

In [19]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from xgboost import XGBRegressor
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error

from app.config import (
    PROCESSED_DRIVER_FEATURES_DIR, INTERIM_RACES_DIR, ARTIFACTS_DIR,
    TRAIN_SEASONS, VAL_SEASONS, TEST_SEASONS
)
from app.models.configs import FINISH_POSITION_MODEL

## Load data

In [20]:
driver_features = pd.concat([pd.read_parquet(f) for f in sorted(PROCESSED_DRIVER_FEATURES_DIR.glob('*.parquet'))])
race_results = pd.concat([pd.read_parquet(f) for f in sorted(INTERIM_RACES_DIR.glob('*.parquet'))])

df = driver_features.merge(
    race_results[['race_id', 'driver_id', 'finish_position', 'grid_position']].rename(columns={'grid_position': 'quali_position'}),
    on=['race_id', 'driver_id'],
    how='left'
)

df = df.dropna(subset=['finish_position'])  # drop DNS/early retirement

print(df.shape)
df.head()

(3455, 26)


,race_id,driver_id,rolling_quali_pos_last_3,rolling_quali_pos_last_5,rolling_finish_pos_last_3,rolling_finish_pos_last_5,rolling_fantasy_points_last_3,rolling_fantasy_points_last_5,rolling_dnf_rate_last_5,circuit_rolling_quali_pos_last_3,...,is_street_circuit,constructor_id,constructor_rolling_fantasy_points_last_3,constructor_rolling_fantasy_points_last_5,constructor_rolling_dnf_rate_last_5,constructor_rolling_quali_pos_last_3,constructor_form_trend_last_5,prediction_stage,finish_position,quali_position
0,2018_1,sebastian_vettel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,ferrari,NaN,NaN,NaN,NaN,NaN,training,1.0,3.0
1,2018_1,lewis_hamilton,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,mercedes,NaN,NaN,NaN,NaN,NaN,training,2.0,1.0
2,2018_1,kimi_räikkönen,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,ferrari,NaN,NaN,NaN,NaN,NaN,training,3.0,2.0
3,2018_1,daniel_ricciardo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,red_bull,NaN,NaN,NaN,NaN,NaN,training,4.0,8.0
4,2018_1,fernando_alonso,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,True,mclaren,NaN,NaN,NaN,NaN,NaN,training,5.0,10.0


## Train/val/test split

In [21]:
config = FINISH_POSITION_MODEL

X = df[config['features']]
y = df[config['target']]

X_train = X[df['season'].isin(TRAIN_SEASONS)]
y_train = y[df['season'].isin(TRAIN_SEASONS)]

X_val = X[df['season'].isin(VAL_SEASONS)]
y_val = y[df['season'].isin(VAL_SEASONS)]

X_test = X[df['season'].isin(TEST_SEASONS)]
y_test = y[df['season'].isin(TEST_SEASONS)]

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

Train: 2497 | Val: 479 | Test: 479


## Baselines

In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# sklearn models need NaN imputation (XGBoost handles it natively)
imputer = SimpleImputer(strategy='mean')

# naive: predict grid position as finish position (no model)
naive_val_pred = X_val['quali_position'].values
naive_test_pred = X_test['quali_position'].values
print('Naive (grid position):')
print(f'  [val]  MAE: {mean_absolute_error(y_val, naive_val_pred):.2f} | Spearman: {spearmanr(y_val, naive_val_pred).statistic:.3f}')
print(f'  [test] MAE: {mean_absolute_error(y_test, naive_test_pred):.2f} | Spearman: {spearmanr(y_test, naive_test_pred).statistic:.3f}')

# linear regression
lr = Pipeline([('imputer', SimpleImputer(strategy='mean')), ('model', LinearRegression())])
lr.fit(X_train, y_train)
print('\nLinear Regression:')
print(f'  [val]  MAE: {mean_absolute_error(y_val, lr.predict(X_val)):.2f} | Spearman: {spearmanr(y_val, lr.predict(X_val)).statistic:.3f}')
print(f'  [test] MAE: {mean_absolute_error(y_test, lr.predict(X_test)):.2f} | Spearman: {spearmanr(y_test, lr.predict(X_test)).statistic:.3f}')

# random forest
rf = Pipeline([('imputer', SimpleImputer(strategy='mean')), ('model', RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42))])
rf.fit(X_train, y_train)
print('\nRandom Forest:')
print(f'  [val]  MAE: {mean_absolute_error(y_val, rf.predict(X_val)):.2f} | Spearman: {spearmanr(y_val, rf.predict(X_val)).statistic:.3f}')
print(f'  [test] MAE: {mean_absolute_error(y_test, rf.predict(X_test)):.2f} | Spearman: {spearmanr(y_test, rf.predict(X_test)).statistic:.3f}')

Naive (grid position):
  [val]  MAE: 2.91 | Spearman: 0.728
  [test] MAE: 3.34 | Spearman: 0.652

Linear Regression:
  [val]  MAE: 2.94 | Spearman: 0.750
  [test] MAE: 3.35 | Spearman: 0.646

Random Forest:
  [val]  MAE: 3.05 | Spearman: 0.741
  [test] MAE: 3.35 | Spearman: 0.658


## XGBoost model

MAE 2.97 (val), 3.32 (test) | Spearman 0.750 (val), 0.661 (test)


In [23]:
model = XGBRegressor(**config['hyperparams'])
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

def evaluate(model, X, y, split_name):
    y_pred = model.predict(X)
    mae = mean_absolute_error(y, y_pred)
    spearman = spearmanr(y, y_pred).statistic
    print(f'[{split_name}] MAE: {mae:.2f} | Spearman: {spearman:.3f}')

print('XGBoost:')
evaluate(model, X_val, y_val, 'val')
evaluate(model, X_test, y_test, 'test')

XGBoost:
[val] MAE: 3.02 | Spearman: 0.746
[test] MAE: 3.36 | Spearman: 0.657


## Feature importance

In [24]:
importance = pd.Series(model.feature_importances_, index=config['features']).sort_values(ascending=False)
print(importance.head(config['feature_importance_top_n']))

constructor_rolling_fantasy_points_last_5    0.267866
quali_position                               0.179215
constructor_rolling_quali_pos_last_3         0.049448
rolling_quali_pos_last_5                     0.047128
rolling_quali_pos_last_3                     0.038291
season_points_to_date                        0.036874
rolling_finish_pos_last_5                    0.036855
rolling_fantasy_points_last_5                0.032482
constructor_rolling_fantasy_points_last_3    0.032307
rolling_fantasy_points_last_3                0.030697
dtype: float32


## Hyperparameter tuning

In [ ]:
# experiment with hyperparams here
tuned_model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=20,
)

tuned_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

evaluate(tuned_model, X_val, y_val, 'val')
evaluate(tuned_model, X_test, y_test, 'test')

[val] MAE: 3.02 | Spearman: 0.746
[test] MAE: 3.36 | Spearman: 0.657


#### Tune max_depth

In [ ]:
for depth in [3, 4, 5, 6]:
    m = XGBRegressor(**{**config['hyperparams'], 'max_depth': depth})
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    print(f'max_depth={depth}:')
    evaluate(m, X_val, y_val, 'val')
    evaluate(m, X_test, y_test, 'test')

max_depth=3:
[val] MAE: 2.97 | Spearman: 0.750
[test] MAE: 3.32 | Spearman: 0.660
max_depth=4:
[val] MAE: 3.02 | Spearman: 0.746
[test] MAE: 3.36 | Spearman: 0.657
max_depth=5:
[val] MAE: 3.07 | Spearman: 0.747
[test] MAE: 3.42 | Spearman: 0.653
max_depth=6:
[val] MAE: 3.09 | Spearman: 0.743
[test] MAE: 3.46 | Spearman: 0.643


#### Tune learning_rate

In [ ]:
for lr in [0.01, 0.05, 0.1, 0.2]:
    m = XGBRegressor(**{**config['hyperparams'], 'max_depth': 3, 'learning_rate': lr})
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    print(f'learning_rate={lr}:')
    evaluate(m, X_val, y_val, 'val')
    evaluate(m, X_test, y_test, 'test')

learning_rate=0.01:
[val] MAE: 3.01 | Spearman: 0.753
[test] MAE: 3.37 | Spearman: 0.664
learning_rate=0.05:
[val] MAE: 2.97 | Spearman: 0.750
[test] MAE: 3.32 | Spearman: 0.660
learning_rate=0.1:
[val] MAE: 2.98 | Spearman: 0.747
[test] MAE: 3.33 | Spearman: 0.657
learning_rate=0.2:
[val] MAE: 3.00 | Spearman: 0.749
[test] MAE: 3.35 | Spearman: 0.660


#### Tune n_estimators

In [30]:
for n in [100, 200, 300, 500]:
    m = XGBRegressor(**{**config['hyperparams'], 'max_depth': 3, 'n_estimators': n})
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    print(f'n_estimators={n}:')
    evaluate(m, X_val, y_val, 'val')
    evaluate(m, X_test, y_test, 'test')

n_estimators=100:
[val] MAE: 2.97 | Spearman: 0.750
[test] MAE: 3.32 | Spearman: 0.661
n_estimators=200:
[val] MAE: 2.97 | Spearman: 0.750
[test] MAE: 3.32 | Spearman: 0.660
n_estimators=300:
[val] MAE: 2.97 | Spearman: 0.750
[test] MAE: 3.32 | Spearman: 0.660
n_estimators=500:
[val] MAE: 2.97 | Spearman: 0.750
[test] MAE: 3.32 | Spearman: 0.660


#### Tune subsample & colsample_byree

In [31]:
for sub, col in [(0.6, 0.6), (0.7, 0.7), (0.8, 0.8), (0.9, 0.9), (1.0, 1.0)]:
    m = XGBRegressor(**{**config['hyperparams'], 'max_depth': 3, 'n_estimators': 100, 'subsample': sub, 'colsample_bytree': col})
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    print(f'subsample={sub}, colsample_bytree={col}:')
    evaluate(m, X_val, y_val, 'val')
    evaluate(m, X_test, y_test, 'test')

subsample=0.6, colsample_bytree=0.6:
[val] MAE: 2.97 | Spearman: 0.754
[test] MAE: 3.33 | Spearman: 0.664
subsample=0.7, colsample_bytree=0.7:
[val] MAE: 2.99 | Spearman: 0.751
[test] MAE: 3.35 | Spearman: 0.661
subsample=0.8, colsample_bytree=0.8:
[val] MAE: 2.97 | Spearman: 0.750
[test] MAE: 3.32 | Spearman: 0.661
subsample=0.9, colsample_bytree=0.9:
[val] MAE: 3.00 | Spearman: 0.752
[test] MAE: 3.34 | Spearman: 0.661
subsample=1.0, colsample_bytree=1.0:
[val] MAE: 2.99 | Spearman: 0.747
[test] MAE: 3.33 | Spearman: 0.661
